# 📝 Cypher 기초 과제 LV2(응용): 다단 패턴·방향·조건 조합·MERGE·결과 다듬기

> 이제 개념을 **조합**합니다. 여러 노드를 잇는 **다단 패턴**, 화살표 **방향**, WHERE 조건과 패턴의 결합, **MERGE** 로 안전하게 등록하기, 그리고 결과를 다듬는 **DISTINCT**·**ORDER BY**·**LIMIT**.

## 풀이 방법
1. 맨 위 **준비 셀 → 초기화 셀 → 시드 셀**을 위에서부터 실행하세요.
2. 각 문제의 **답안 셀**을 채우고 **자가채점 셀**로 확인하세요(✅ 통과!).
3. **문제 번호 순서대로 푸세요.** 6·7번은 그래프에 회원·관계를 **더하는** 문제라, 먼저 풀면 앞 문제의 채점 결과가 달라져 떨어집니다(8번은 7번이 만든 관계를 조회합니다). **9~11번은 6·7번이 등록한 정수빈까지 포함해** 답을 냅니다.

- 도메인: 피트니스 센터 **코어짐**: 회원(`Member`), 수업(`Class`: `level`), 강사(`Trainer`: `specialty`). 관계는 `ATTENDS`(회원→수업)·`TAUGHT_BY`(수업→강사)입니다.
- 수강 관계 `ATTENDS` 에는 **등록 개월 수 `months`** 가 담겨 있습니다(회원의 값도 수업의 값도 아니라 **그 수강 하나의 값**이라 관계에 붙어 있습니다).
- 다단 패턴은 화살표를 이어 그립니다. 회원에서 `ATTENDS` 로 수업에, 수업에서 `TAUGHT_BY` 로 강사에 이어지는 식입니다(정확한 패턴 문법은 교안을 참고하세요).
- **주의**: 한 회원이 조건에 맞는 수업을 여러 개 들으면 결과에 이름이 **여러 번** 나올 수 있습니다. 그런 문제는 파이썬 **집합(set)**으로 중복을 없애 비교하세요.

화이팅!

아래 준비 셀 3개를 먼저 실행하세요.

In [ ]:
# [제공 코드] Neo4j 연결: 실행만 하세요. 반드시 "실습 전용" DB 여야 합니다(아래 실습이 그래프를 지웁니다).
# 앞으로 모든 Cypher 는 run_cypher("쿼리", 파라미터=값) 으로 실행하고, 결과는 dict 리스트로 옵니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase
from neo4j.exceptions import ConstraintError  # 지우기 규칙 위반 에러

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# .env 를 못 읽어도 에러 없이 기본값으로 넘어간다. 마지막 줄에 찍히는 주소를 눈으로 꼭 확인할 것
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)   # 여기까지 찍히면 준비 완료

> ⚠️ **아래 초기화 셀은 연결된 데이터베이스의 노드를 전부 지웁니다.** 지난 단원에서 브라우저로 적재한 **Movies 예제 그래프와 그때 푼 과제 결과도 함께 사라집니다.** 되돌릴 수 없으니, `.env` 가 **실습 전용 DB** 를 가리키는지 먼저 확인하세요. Movies 를 남기고 싶다면 실습용 인스턴스를 따로 하나 만들어 그 접속 정보를 `.env` 에 넣으면 됩니다(지웠더라도 day28 폴더의 `data/movies_setup.cypher` 로 다시 적재할 수 있습니다).

In [ ]:
# [제공 코드] 그래프 초기화: 실습 전용 DB 인지 꼭 확인하고 실행하세요! 노드·관계를 전부 지웁니다.
# MATCH (n) 은 모든 노드, DETACH 는 붙어 있는 관계까지 함께 지우라는 뜻입니다(교안_01 마지막 절에 나옵니다).
run_cypher("MATCH (n) DETACH DELETE n")
# 확인: MATCH (n) RETURN n 은 남은 노드를 한 줄씩 돌려주므로 그 행 수가 곧 노드 개수다
print("초기화 완료. 남은 노드:", len(run_cypher("MATCH (n) RETURN n")))

In [ ]:
# [제공 코드] 피트니스 센터 "코어짐" 그래프 적재: 이 셀은 실행만 하세요.
# 회원(Member)·수업(Class: level)·강사(Trainer: specialty)와 수강(ATTENDS)·담당(TAUGHT_BY) 관계를 만듭니다.
trainers = [("강타쿠", "웨이트"), ("박유연", "요가"), ("이러닝", "러닝")]
classes = [
    ("파워리프팅", "고급", "강타쿠"),
    ("상체근력", "중급", "강타쿠"),
    ("하타요가", "초급", "박유연"),
    ("플라잉요가", "중급", "박유연"),
    ("트레일런", "중급", "이러닝"),
]
# 회원이 어떤 수업을 몇 개월째 듣고 있는지까지 적어 둔다(개월 수는 수강 관계에 담긴다)
members = [
    ("홍길동", [("파워리프팅", 12), ("하타요가", 3)]),
    ("김영희", [("상체근력", 6), ("플라잉요가", 9)]),
    ("이철수", [("트레일런", 24), ("파워리프팅", 4)]),
    ("박민수", [("하타요가", 1)]),
    ("최지아", [("플라잉요가", 18), ("트레일런", 2)]),
]

for name, spec in trainers:
    run_cypher("CREATE (:Trainer {name: $name, specialty: $spec})", name=name, spec=spec)
for name, level, trainer in classes:
    run_cypher("CREATE (:Class {name: $name, level: $level})", name=name, level=level)
    run_cypher(
        "MATCH (c:Class {name: $name}), (t:Trainer {name: $trainer}) "
        "CREATE (c)-[:TAUGHT_BY]->(t)",
        name=name, trainer=trainer,
    )
for name, plans in members:
    run_cypher("CREATE (:Member {name: $name})", name=name)
    for cn, months in plans:
        run_cypher(
            "MATCH (m:Member {name: $name}), (c:Class {name: $cn}) "
            "CREATE (m)-[:ATTENDS {months: $months}]->(c)",
            name=name, cn=cn, months=months,
        )

print("적재한 회원 수:", len(run_cypher("MATCH (m:Member) RETURN m")))


이 과제가 쓰는 그래프의 구조입니다. 레이블·속성과 관계의 **방향**을 먼저 확인하세요.

<img src="images/gym-schema.png" width="820">

## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다. 어떤 수업을 누가 가르치는지 먼저 훑어봅니다.

In [ ]:
# [제공 코드] 수업과 담당 강사를 먼저 살펴봅니다
for r in run_cypher(
    "MATCH (c:Class)-[:TAUGHT_BY]->(t:Trainer) RETURN c.name AS cls, c.level AS level, t.name AS trainer"
):
    print(r['cls'], '(', r['level'], ') -', r['trainer'])

수강 관계에는 **등록 개월 수 `months`** 가 붙어 있습니다. 아래 셀도 **실행만** 하면 됩니다.

In [ ]:
# [제공 코드] 누가 어떤 수업을 몇 개월째 듣고 있는지 살펴봅니다
for r in run_cypher(
    "MATCH (m:Member)-[r:ATTENDS]->(c:Class) "
    "RETURN m.name AS member, c.name AS cls, r.months AS months"
):
    print(r['member'], '-', r['cls'], ':', r['months'], '개월')

## 1. 회원이 만나는 강사 (다단 패턴)
**배경**: **홍길동** 회원이 듣는 수업들의 **담당 강사**가 누구인지 봅니다.

**요구사항**:
- 홍길동 회원에서 시작해 `ATTENDS`(회원→수업), 이어서 `TAUGHT_BY`(수업→강사)를 따라가 강사까지 이어지는 **체인 패턴**을 MATCH 하세요. 강사 이름을 별칭 **`name`** 으로 RETURN 해 결과를 변수 **`hong_trainers`** 에 담으세요.

**예시**: 홍길동이 만나는 강사는 **2명** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 회원에서 수업으로, 다시 수업에서 강사로 두 관계를 한 방향으로 이어 간다.

세부구현:
1. MATCH 에서 홍길동을 시작점으로 삼아 ATTENDS 와 TAUGHT_BY 를 차례로 이어 강사 노드까지 간다.
2. 강사 이름을 별칭 name 으로 RETURN 해 hong_trainers 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in hong_trainers) == ['강타쿠', '박유연'], (
    '홍길동이 만나는 강사가 다릅니다. ATTENDS 다음에 TAUGHT_BY 를 이어 강사까지 갔는지, '
    '별칭이 name 인지 확인하세요'
)
print('✅ 통과!')

## 2. 강사의 수업을 듣는 회원 (역방향 패턴)
**배경**: 이번엔 반대로, **강타쿠** 강사의 수업을 듣는 **회원**들을 찾습니다.

**요구사항**:
- 강타쿠 강사에서 시작해 화살표를 **거꾸로** 따라갑니다: `TAUGHT_BY` 를 거슬러 수업으로, 다시 `ATTENDS` 를 거슬러 회원까지. 회원 이름을 별칭 **`name`** 으로 RETURN 해 결과를 변수 **`kang_members`** 에 담으세요.

**예시**: 강타쿠의 수업을 듣는 회원은 **3명** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 관계 종류는 그대로 두고 화살표 방향만 반대로 그려, 강사에서 수업을 거쳐 회원으로 거슬러 올라간다.

세부구현:
1. MATCH 에서 강타쿠를 시작점으로, TAUGHT_BY 와 ATTENDS 를 거꾸로 따라가 회원 노드까지 간다.
2. 회원 이름을 별칭 name 으로 RETURN 해 kang_members 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in kang_members) == ['김영희', '이철수', '홍길동'], \
    '강타쿠의 수업을 듣는 회원이 다릅니다. 강사에서 출발해 화살표를 거꾸로(<-) 그렸는지 확인하세요'
print('✅ 통과!')

## 3. 중급 수업을 듣는 회원 (WHERE + 패턴)
**배경**: 난이도 **중급** 수업을 듣는 회원이 누구인지 봅니다.

**요구사항**:
- 회원이 `ATTENDS` 로 수업에 이어지는 패턴을 MATCH 하고, 수업의 `level` 이 `'중급'` 인 조건을 `WHERE` 로 건 뒤, 회원 이름을 별칭 **`name`** 으로 RETURN 해 변수 **`mid_members`** 에 담으세요.
- 한 회원이 중급 수업을 **여러 개** 들으면 여러 번 나올 수 있으니, 채점은 **집합**으로 합니다.

**예시**: 중급 수업을 듣는 회원은(중복 제거) **3명** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 회원에서 수업으로 가는 패턴을 만들고, WHERE 로 수업 난이도 조건을 건다.

세부구현:
1. MATCH 로 회원(Member)이 ATTENDS 로 수업(Class)에 이어지는 패턴을 만든다.
2. WHERE 로 수업의 level 이 중급인 조건을 건다.
3. 회원 이름을 별칭 name 으로 RETURN 해 mid_members 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert sorted({r['name'] for r in mid_members}) == ['김영희', '이철수', '최지아'], \
    '중급 수업을 듣는 회원이 다릅니다. WHERE 로 수업의 level 을 중급으로 좁혔는지 확인하세요'
print('✅ 통과!')

## 4. 박유연 강사의 초급 수업 수강생 (다단 + WHERE)
**배경**: **박유연** 강사가 담당하는 수업 중 **초급** 수업을 듣는 회원을 찾습니다.

**요구사항**:
- 회원이 `ATTENDS` 로 수업에, 그 수업이 `TAUGHT_BY` 로 강사에 이어지는 체인 패턴에서 강사를 `박유연` 으로 고정하고, `WHERE` 로 수업의 `level` 이 `'초급'` 인 조건을 건 뒤, 회원 이름을 별칭 **`name`** 으로 RETURN 해 변수 **`park_basic`** 에 담으세요.

**예시**: 조건을 만족하는 회원은 **2명** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 회원에서 수업, 수업에서 강사로 이어지는 체인에서 강사를 박유연으로 고정하고, WHERE 로 난이도를 초급으로 좁힌다.

세부구현:
1. MATCH 로 회원→수업→강사 체인을 만들되, 강사(Trainer)의 이름을 박유연으로 고정한다.
2. WHERE 로 수업의 level 이 초급인 조건을 건다.
3. 회원 이름을 별칭 name 으로 RETURN 해 park_basic 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in park_basic) == ['박민수', '홍길동'], (
    '조건을 만족하는 회원이 다릅니다. 강사를 박유연으로 고정하고 WHERE 로 level 을 초급으로 '
    '좁혔는지 확인하세요. 6번(신규 회원 등록)을 먼저 풀었다면 그 회원이 더해지니, 맨 위 '
    '초기화 셀과 시드 셀부터 다시 실행한 뒤 순서대로 푸세요'
)
print('✅ 통과!')

## 5. 두 회원이 함께 듣는 수업 (공유 패턴)
**배경**: **홍길동**과 **이철수**가 **함께** 듣는 수업이 있는지 봅니다.

**요구사항**:
- 두 회원 홍길동·이철수가 **가운데 수업을 공유**하는 패턴을 MATCH 하세요. 두 회원이 각각 `ATTENDS` 로 같은 수업을 향하도록(화살표가 마주 봄) 잇습니다. 공유 수업 이름을 별칭 **`name`** 으로 RETURN 해 변수 **`shared`** 에 담으세요.

**예시**: 두 사람이 함께 듣는 수업은 **1개** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 두 회원이 같은 수업 노드를 향하도록, 가운데 수업을 공유하는 패턴을 그린다(화살표가 마주 봄).
- 교안의 "같은 프로젝트 동료" 공유 패턴과 같은 모양이다.

세부구현:
1. 한쪽 회원에서 ATTENDS 로 수업으로 가고, 그 수업으로 다른 회원이 ATTENDS 로 들어오게 잇는다.
2. 두 회원 이름은 각각 홍길동·이철수로 고정한다.
3. 가운데 수업 이름을 별칭 name 으로 RETURN 해 shared 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in shared) == ['파워리프팅'], (
    '함께 듣는 수업이 다릅니다. 두 회원이 가운데 수업 하나를 함께 향하도록(화살표가 마주 봄) '
    '이었는지 확인하세요'
)
print('✅ 통과!')

## 6. 새 회원 등록하기 (MERGE 노드·관계)
**배경**: 신규 회원 **정수빈**이 **하타요가** 수업을 신청했습니다. 등록 코드를 실수로 두 번 실행해도 중복이 안 생기게 합니다. 이 문제는 그래프에 회원을 **더하므로 맨 마지막에** 둡니다(앞 문제들을 먼저 푸세요).

**요구사항**:
- 실행할 Cypher 문 **두 개**를 각각 문자열 변수 **`member_q`**(정수빈 `Member` 노드를 `MERGE`)와 **`attend_q`**(정수빈과 하타요가 `Class` 를 `MATCH` 로 찾아 그 사이 `ATTENDS` 관계를 `MERGE`)에 담고, `run_cypher` 로 각각 실행하세요.
- **자가채점 셀이 두 문을 한 번 더 실행**합니다. 그래도 정수빈 노드도, 수강 관계도 **하나씩**이어야 통과합니다(`CREATE` 로 쓰면 재실행에서 늘어나 떨어집니다).

**예시**: 채점이 재실행한 뒤에도 정수빈 노드 수·수강 관계 수는 각각 **1** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 노드도 관계도 CREATE 대신 MERGE 로 만들어 반복 실행에도 중복이 안 생기게 한다.

세부구현:
1. 정수빈 회원 노드를 MERGE 하는 문을 문자열 변수 member_q 에 담는다.
2. 정수빈과 하타요가를 MATCH 로 찾아 그 사이 ATTENDS 관계를 MERGE 하는 문을 attend_q 에 담는다.
3. run_cypher 로 두 문을 각각 실행한다(채점 셀이 같은 두 문을 한 번 더 실행한다).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
for q in (member_q, attend_q):
    up = q.upper()
    assert 'MERGE' in up and 'CREATE' not in up, (
        'member_q 와 attend_q 는 둘 다 MERGE 문이어야 합니다. CREATE 로 쓰면 채점이 한 번 더 '
        '실행할 때 회원이나 수강 관계가 두 개가 됩니다'
    )
    run_cypher(q)   # 채점이 한 번 더 실행: 멱등이면 늘지 않는다
assert len(run_cypher("MATCH (m:Member {name: '정수빈'}) RETURN m")) == 1, (
    '정수빈 회원이 하나가 아닙니다. member_q 에 CREATE 를 썼다면 맨 위 초기화 셀과 시드 셀부터 '
    '다시 실행하세요'
)
assert len(run_cypher(
    "MATCH (:Member {name: '정수빈'})-[:ATTENDS]->(:Class {name: '하타요가'}) RETURN 1 AS x"
)) == 1, (
    '정수빈의 하타요가 수강 관계가 하나가 아닙니다. attend_q 에 CREATE 를 썼다면 맨 위 초기화 '
    '셀과 시드 셀부터 다시 실행하세요'
)
print('✅ 통과!')

## 7. 수강 정보를 관계에 담아 등록하기 (CREATE 관계 속성)
**배경**: 6번에서 등록한 **정수빈**이 **트레일런** 수업도 신청했습니다. 시드의 다른 수강에 `months` 가 담겨 있듯이, 이번에는 **언제부터**(`since`) **주 몇 회**(`times`) 나오는지를 이번 수강에 남깁니다. 이 값들은 회원의 값도 수업의 값도 아니라 **그 수강 하나의 값**이라 관계에 담습니다. (6번을 먼저 풀어야 정수빈이 있습니다.)

**요구사항**:
- 정수빈(`Member`)과 트레일런(`Class`)을 각각 `MATCH` 로 찾아 그 사이에 `ATTENDS` 관계(회원→수업)를 `CREATE` 하되, 관계에 변수 `r` 과 속성 **`{since: 2026, times: 3}`** 을 함께 적으세요.
- **같은 문장 끝에 `RETURN`** 을 이어 써 `r.since`·`r.times` 를 별칭 **`since`**·**`times`** 로 돌려받고, 그 결과를 변수 **`subin_attend`** 에 담아 출력하세요(다시 `MATCH` 하지 말고 **만드는 그 문장에서** 돌려받아야 합니다).
- **이 답안 셀은 한 번만 실행하세요**(`CREATE` 라 두 번 실행하면 수강 관계가 두 개가 됩니다). 그리고 **7·8번은 맨 마지막에** 푸세요. 그래프에 관계를 더하므로 앞 문제보다 먼저 풀면 앞 문제의 채점이 떨어집니다.

**예시**: `subin_attend` 가 `[{'since': 2026, 'times': 3}]` 한 줄이면 됩니다.

<details><summary>힌트</summary>

```text
접근방법:
- 관계 속성은 노드 속성과 똑같이 {키: 값} 인데, 자리만 대괄호 안이다.

세부구현:
1. 정수빈과 트레일런을 쉼표로 나란히 MATCH 한다.
2. 그 사이 관계를 CREATE 하되 대괄호 안에 변수 r 과 두 속성을 함께 적는다.
3. 같은 문장 끝에 RETURN 을 이어 써 두 값을 별칭 since·times 로 돌려받아 subin_attend 에 담고 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 1) 그래프에 관계와 두 속성이 제대로 들어갔는지
rows = run_cypher(
    "MATCH (:Member {name: '정수빈'})-[r:ATTENDS]->(:Class {name: '트레일런'}) "
    "RETURN r.since AS since, r.times AS times"
)
assert len(rows) == 1, (
    '정수빈-트레일런 수강 관계가 하나가 아닙니다. 답안 셀을 두 번 실행했다면 맨 위 초기화 셀과 '
    '시드 셀부터 다시 실행한 뒤 1번부터 순서대로 푸세요'
)
assert rows[0]['since'] == 2026 and rows[0]['times'] == 3, \
    '관계 속성 값이 다릅니다. 대괄호 안에 {since: 2026, times: 3} 을 적었는지 확인하세요'
# 2) 만드는 그 문장에서 RETURN 으로 돌려받았는지(subin_attend 가 그 증거다)
assert subin_attend == [{'since': 2026, 'times': 3}], (
    'subin_attend 가 다릅니다. CREATE 문 끝에 RETURN r.since AS since, r.times AS times 를 이어 써 '
    '그 결과를 subin_attend 에 담으세요(다시 MATCH 해서 받은 값이 아니라 만드는 문장이 돌려준 값입니다)'
)
print('✅ 통과!')

## 8. 주 3회 이상 수업의 담당 강사 찾기 (관계 속성 조건 + 다단 패턴)
**배경**: 정수빈이 자주 나오는 수업을 누가 가르치는지 봅니다. **횟수는 관계에 붙은 값**이고 **강사는 이웃 노드**라, 조건을 거는 자리가 서로 다릅니다. (7번을 먼저 풀어야 합니다.)

**요구사항**:
- 회원에서 `ATTENDS` 로 수업에, 그 수업에서 `TAUGHT_BY` 로 강사에 이어지는 **체인 패턴**을 MATCH 하되, 회원을 `정수빈` 으로 고정하고 **`ATTENDS` 관계에 변수 `r`** 을 붙이세요.
- `WHERE r.times >= 3` 으로 **주 3회 이상**인 수강만 남기세요.
- **수업 이름**을 별칭 **`name`**, **강사 이름**을 별칭 **`trainer`** 로 **함께** RETURN 하고, 그 결과를 변수 **`subin_plan`** 에 담으세요.

**예시**: `subin_plan` 은 **한 줄**이고, 그 행에 `name`·`trainer` 두 키가 있습니다. 6번에서 만든 하타요가 수강 관계에는 `times` 가 없어 비교가 성립하지 않으므로 자연히 빠집니다.

<details><summary>힌트</summary>

```text
접근방법:
- 조건이 두 자리에 나뉜다: 횟수는 관계에 붙은 값이라 WHERE, 강사는 이웃 노드라 패턴.

세부구현:
1. 회원에서 ATTENDS 로 수업에, 수업에서 TAUGHT_BY 로 강사에 이어지는 패턴을 MATCH 하되 ATTENDS 에 변수 r 을 붙인다.
2. 회원 이름을 정수빈으로 고정한다.
3. WHERE 로 r 의 times 조건을 걸고, 수업 이름과 강사 이름을 별칭 name·trainer 로 RETURN 해 subin_plan 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert all(isinstance(r, dict) and {'name', 'trainer'} <= set(r) for r in subin_plan), (
    'subin_plan 에는 run_cypher 결과를 그대로 담으세요(dict 의 리스트입니다). '
    '각 행에 별칭 name(수업 이름)과 trainer(강사 이름)가 함께 있어야 합니다'
)
assert sorted((r['name'], r['trainer']) for r in subin_plan) == [('트레일런', '이러닝')], (
    '결과가 다릅니다. 별칭 이름(name 은 수업, trainer 는 강사), 화살표 방향, 그리고 WHERE 에 건 '
    '조건이 관계 변수 r 의 times 인지 확인하세요'
)
print('✅ 통과!')

## 9. 중급이 아닌 수업과 그 강사 (NOT + 다단 패턴)
**배경**: 입문반과 상급반을 따로 안내하려고 **중급이 아닌** 수업을 듣는 회원과 그 담당 강사를 함께 뽑습니다. 3번은 중급 수업을 **듣는** 회원이었죠. 이번엔 그 조건을 `NOT` 으로 뒤집고, 강사까지 한 문장으로 이어 갑니다. (**6번까지 푼 상태**여야 합니다.)

**요구사항**:
- 회원이 `ATTENDS` 로 수업에, 그 수업이 `TAUGHT_BY` 로 강사에 이어지는 **체인 패턴**을 MATCH 하세요.
- `WHERE` 에 **`NOT`** 을 써서 수업의 `level` 이 `'중급'` 인 조건을 **뒤집으세요**.
- **회원 이름**을 별칭 **`name`**, **강사 이름**을 별칭 **`trainer`** 로 **함께** RETURN 하고, 그 결과를 변수 **`not_mid`** 에 담으세요.

**예시**: 결과는 **5줄** 입니다. 한 회원이 중급이 아닌 수업을 둘 들으면 **두 줄로 나옵니다**(패턴 곱). 여기서는 중복을 없애지 않고 그대로 둡니다. 없애는 방법은 바로 다음 10번에서 다룹니다.

<details><summary>힌트</summary>

```text
접근방법:
- 4번에서 쓰던 회원-수업-강사 체인을 그대로 쓴다. 바뀌는 것은 WHERE 의 조건뿐이다.
- NOT 은 뒤에 오는 조건을 통째로 뒤집는다.

세부구현:
1. MATCH 로 회원에서 ATTENDS 로 수업에, 수업에서 TAUGHT_BY 로 강사에 이어지는 패턴을 만든다.
2. WHERE 에 NOT 을 붙여 수업의 level 이 중급인 조건을 뒤집는다.
3. 회원 이름과 강사 이름을 별칭 name·trainer 로 함께 RETURN 해 not_mid 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert all(isinstance(r, dict) and {'name', 'trainer'} <= set(r) for r in not_mid), (
    'not_mid 에는 run_cypher 결과를 그대로 담으세요(dict 의 리스트입니다). '
    '각 행에 별칭 name(회원)과 trainer(강사)가 함께 있어야 합니다'
)
assert sorted((r['name'], r['trainer']) for r in not_mid) == [('박민수', '박유연'), ('이철수', '강타쿠'), ('정수빈', '박유연'), ('홍길동', '강타쿠'), ('홍길동', '박유연')], (
    '결과가 다릅니다. WHERE 에 NOT 을 붙여 level 조건을 뒤집었는지, 별칭 이름(name 은 회원, '
    'trainer 는 강사)이 맞는지, 6번까지 풀었는지 확인하세요. 중복 행은 그대로 두어야 합니다'
)
print('✅ 통과!')

## 10. 중급 수업을 듣는 회원, 중복 없이 (DISTINCT)
**배경**: 3번과 **같은 질문**입니다. 그때는 이름이 여러 번 나오는 것을 파이썬 **집합**으로 지웠습니다. 이번에는 그 일을 **Cypher 가** 하게 합니다. (6·7번에서 등록한 정수빈도 중급 수업(트레일런)을 듣고 있어 3번 때보다 한 명 늘어납니다.)

**요구사항**:
- 회원이 `ATTENDS` 로 수업에 이어지는 패턴을 MATCH 하고, `WHERE` 로 수업의 `level` 이 `'중급'` 인 것만 남긴 뒤, 회원 이름을 별칭 **`name`** 으로 RETURN 해 변수 **`mid_unique`** 에 담으세요.
- 같은 이름이 여러 줄 나오지 않도록 **`DISTINCT`** 로 중복을 없애세요(`RETURN DISTINCT ...` 처럼 RETURN 바로 뒤에 붙입니다).
- **파이썬 집합으로 지우지 마세요.** 채점은 `mid_unique` 의 **줄 수**를 봅니다.

**예시**: `mid_unique` 는 **4줄** 입니다(중복을 없애지 않으면 6줄이 나옵니다).

<details><summary>힌트</summary>

```text
접근방법:
- 3번과 똑같은 MATCH·WHERE 를 쓰고, 중복만 Cypher 쪽에서 없앤다.

세부구현:
1. MATCH 로 회원이 ATTENDS 로 수업에 이어지는 패턴을 만든다.
2. WHERE 로 수업의 level 이 중급인 것만 남긴다.
3. RETURN 바로 뒤에 DISTINCT 를 붙이고 회원 이름을 별칭 name 으로 돌려받아 mid_unique 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert all(isinstance(r, dict) and 'name' in r for r in mid_unique), (
    'mid_unique 에는 run_cypher 결과를 그대로 담으세요(dict 의 리스트입니다). '
    '파이썬 집합으로 중복을 지우면 이 검사에서 걸립니다. 중복은 Cypher 의 DISTINCT 로 '
    '없애세요'
)
assert len(mid_unique) == 4, (
    'mid_unique 가 4줄이 아닙니다. RETURN 뒤에 DISTINCT 를 붙여 중복을 없앴는지 '
    '확인하세요(없애지 않으면 6줄입니다). 파이썬 집합으로 지우지 말고 Cypher 가 '
    '지우게 하세요. 6·7번까지 풀었는지도 확인하세요'
)
assert sorted(r['name'] for r in mid_unique) == ['김영희', '이철수', '정수빈', '최지아'], (
    '회원 이름이 다릅니다. WHERE 로 수업의 level 을 중급으로 좁혔는지, 별칭이 name 인지 '
    '확인하세요'
)
print('✅ 통과!')

## 11. 가장 오래 다닌 수강 3건 (ORDER BY·LIMIT + IS NOT NULL)
**배경**: 오래 다닌 회원에게 감사 쿠폰을 보내려고 **등록 개월 수가 긴 수강 3건**을 뽑습니다. 6·7번에서 등록한 정수빈의 수강 두 건에는 `months` 가 없다는 점에 주의하세요.

**요구사항**:
- 회원이 `ATTENDS` 로 수업에 이어지는 패턴을 MATCH 하되 **관계에 변수 `r`** 을 붙이세요.
- `WHERE` 에 **`IS NOT NULL`** 을 써서 `r.months` 가 **적혀 있는** 수강만 남기세요.
- 회원 이름을 별칭 **`name`**, 수업 이름을 별칭 **`cls`**, 개월 수를 별칭 **`months`** 로 **함께** RETURN 하고, **`ORDER BY`** 로 개월 수가 **큰 값부터**(내림차순 `DESC`) 줄 세운 뒤 **`LIMIT`** 으로 앞 **3줄**만 남기세요.
- 실행할 Cypher 문을 **문자열 변수 `rank_q`** 에 담고, **`top_plans = run_cypher(rank_q)`** 로 실행하세요. **자가채점 셀이 `rank_q` 를 한 번 더 실행**해, 그 문장 하나만으로 답이 나오는지 확인합니다.
- 그래서 **파이썬에서 정렬하거나 잘라내면 통과하지 못합니다.** 거르기·줄 세우기·끊기를 모두 Cypher 안에서 하세요(순서까지 채점합니다).

**예시**: 1위는 **24개월** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 정렬 기준이 관계에 붙은 값이므로 관계에 변수를 붙여야 꺼낼 수 있다.
- 값이 비어 있는 줄을 먼저 걸러 낸 다음 줄을 세우고, 마지막에 앞에서 끊는다.
- ORDER BY 에는 RETURN 에서 붙인 별칭을 그대로 써도 된다.

세부구현:
1. MATCH 로 회원이 ATTENDS 로 수업에 이어지는 패턴을 만들되 관계에 변수 r 을 붙인다.
2. WHERE 로 r 의 months 가 비어 있지 않은 수강만 남긴다.
3. 회원 이름·수업 이름·개월 수를 각각 별칭 name·cls·months 로 함께 RETURN 한다.
4. ORDER BY 로 개월 수를 내림차순 정렬하고 LIMIT 으로 앞 3줄만 남긴다.
5. 그 문장을 문자열 변수 rank_q 에 담고 run_cypher 에 넘겨 결과를 top_plans 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
up = rank_q.upper()
assert 'ORDER BY' in up, (
    'rank_q 에 ORDER BY 가 없습니다. 줄 세우기는 파이썬이 아니라 Cypher 가 해야 합니다'
)
assert 'LIMIT' in up, (
    'rank_q 에 LIMIT 이 없습니다. 앞에서 3줄만 남기는 것도 Cypher 안에서 하세요'
)
again = run_cypher(rank_q)   # 채점이 같은 문을 한 번 더 실행: 그 문장만으로 답이 나와야 한다
assert len(again) == 3, (
    'rank_q 가 돌려주는 줄이 3줄이 아닙니다. LIMIT 으로 앞에서 3줄만 남겼는지 확인하세요'
)
assert all(isinstance(r, dict) and {'name', 'cls', 'months'} <= set(r) for r in again), (
    '각 행에 별칭 name(회원)·cls(수업)·months(개월 수)가 함께 있어야 합니다'
)
assert [(r['name'], r['cls'], r['months']) for r in again] == [('이철수', '트레일런', 24), ('최지아', '플라잉요가', 18), ('홍길동', '파워리프팅', 12)], (
    '3건이나 그 순서가 다릅니다. ORDER BY 에 DESC 를 붙였는지, WHERE 에 IS NOT NULL 로 개월 수가 '
    '비어 있는 수강을 걸러 냈는지 확인하세요(걸러 내지 않으면 값이 없는 줄이 맨 앞으로 옵니다)'
)
assert [(r['name'], r['cls'], r['months']) for r in top_plans] == [('이철수', '트레일런', 24), ('최지아', '플라잉요가', 18), ('홍길동', '파워리프팅', 12)], (
    'top_plans 에는 run_cypher(rank_q) 의 결과를 그대로 담으세요(파이썬에서 다시 정렬하거나 '
    '잘라내지 마세요)'
)
print('✅ 통과!')

## 12. 새 수업 등록에 처음·재확인 표시 남기기 (ON CREATE SET·ON MATCH SET)
**배경**: 6번에서 `MERGE` 로 안전하게 등록하는 법을 익혔습니다. 그런데 `MERGE` 패턴 안에는 **식별 속성만** 적어야 하죠(다른 값까지 적으면 값이 바뀔 때마다 노드가 새로 생깁니다). 그럼 나머지 값은 언제 채울까요. **처음 만들 때만** 남길 값과 **다시 마주칠 때마다** 갱신할 값이 서로 다릅니다.

**요구사항**: 새 수업 **모닝필라테스**를 등록합니다.
- 실행할 Cypher 문을 **문자열 변수 `class_q`** 에 담으세요. `MERGE (c:Class {name: '모닝필라테스'})` 처럼 패턴에는 **식별 속성 `name` 만** 넣습니다.
- 거기에 **`ON CREATE SET`** 으로 `c.opened` 를 `2026` 으로, **`ON MATCH SET`** 으로 `c.checked` 를 `1` 로 채우도록 이어 쓰세요(둘 다 쓸 때는 `ON CREATE` 를 먼저 적습니다).
- 그 문장을 `run_cypher(class_q)` 로 **두 번 실행**하세요. 첫 실행은 만드는 쪽, 두 번째 실행은 찾는 쪽으로 갈라집니다.
- 이어서 모닝필라테스의 `opened` 와 `checked` 를 각각 별칭 **`opened`**·**`checked`** 로 함께 RETURN 하고, **첫 줄 하나만**(`[0]`) 변수 **`morning`** 에 담아 출력하세요.

**예시**: `morning` 이 `{'opened': 2026, 'checked': 1}` 이면 됩니다. 한 번만 실행하면 `checked` 가 `None` 인 채로 남습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 같은 문장이 첫 실행과 두 번째 실행에서 서로 다른 쪽으로 갈라진다. 그래서 문자열을 변수에 담아 둔다.
- 패턴에 값을 더 적으면 '같은 것'의 기준이 바뀌어 노드가 새로 생긴다. 그래서 name 만 남긴다.

세부구현:
1. MERGE (c:Class {name: '모닝필라테스'}) 뒤에 ON CREATE SET 과 ON MATCH SET 을 차례로 이어 쓴다.
2. 그 문자열을 class_q 에 담고 run_cypher 로 두 번 실행한다.
3. 모닝필라테스를 MATCH 해 opened·checked 를 별칭 그대로 RETURN 하고 [0] 으로 첫 줄만 꺼내 morning 에 담아 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
up = class_q.upper()
assert 'MERGE' in up, (
    'class_q 는 MERGE 문이어야 합니다. CREATE 로 쓰면 두 번 실행할 때 수업이 두 개가 됩니다'
)
assert 'ON CREATE SET' in up and 'ON MATCH SET' in up, (
    'class_q 에 ON CREATE SET 과 ON MATCH SET 이 둘 다 있어야 합니다. 처음 만들 때와 다시 만났을 '
    '때를 갈라 적는 것이 이 문제입니다'
)
# MERGE 패턴은 여는 괄호부터 첫 닫는 괄호까지다. 그 안에 식별 속성 말고 다른 값이 있으면 안 된다
merge_pattern = class_q[up.find('MERGE'):]
merge_pattern = merge_pattern[:merge_pattern.find(')') + 1]
assert 'opened' not in merge_pattern and 'checked' not in merge_pattern, (
    'MERGE 패턴 안에는 식별 속성 name 만 넣으세요. opened·checked 까지 적으면 그 값이 '
    '"같은 것" 의 기준이 되어 값이 달라질 때마다 노드가 새로 생깁니다(교안_02 4-2)'
)
run_cypher(class_q)   # 채점이 한 번 더 실행: 멱등이면 수업 수도 opened 도 그대로여야 한다
assert len(run_cypher("MATCH (c:Class {name: '모닝필라테스'}) RETURN c")) == 1, (
    '모닝필라테스가 하나가 아닙니다. 패턴 안에 name 말고 다른 값까지 적었다면 값이 다를 때마다 '
    '새 노드가 생깁니다(맨 위 초기화 셀과 시드 셀부터 다시 실행한 뒤 순서대로 푸세요)'
)
assert morning == {'opened': 2026, 'checked': 1}, (
    'morning 이 다릅니다. class_q 를 두 번 실행했는지, opened 는 ON CREATE SET 에 checked 는 '
    'ON MATCH SET 에 적었는지, 별칭이 opened·checked 인지 확인하세요([0] 으로 첫 줄만 담습니다)'
)
assert run_cypher(
    "MATCH (c:Class {name: '모닝필라테스'}) RETURN c.opened AS opened"
)[0]['opened'] == 2026, (
    'opened 가 덮어써졌습니다. 처음 만들 때만 남겨야 하는 값이므로 ON CREATE SET 쪽에 적어야 합니다'
)
print('✅ 통과!')

## 13. 수업 목록을 한 쿼리로 적재하기 (`$이름` 자리표시자)
**배경**: 12번까지는 수업 이름을 쿼리 안에 **글자 그대로** 적었습니다. 그런데 적재할 이름이 파이썬 리스트에 들어 있다면요. 이름마다 쿼리 문자열을 새로 조립하는 것은 번거롭고, 이름에 따옴표라도 들어 있으면 쿼리가 깨집니다. 쿼리는 그대로 두고 **값만 따로 넘기는** 자리가 자리표시자입니다. (**12번을 먼저 푼 상태**여야 합니다.)

**요구사항**:
- 쿼리 문자열을 변수 **`class_merge_q`** 에 담되, 이름 자리에 값을 적지 말고 자리표시자 **`$name`** 을 두세요(`Class` 노드를 **`MERGE`** 하는 문입니다).
- 적재할 이름 세 개를 리스트 변수 **`new_classes`** 에 `['모닝필라테스', "Rock'n'Roll 스피닝", '하타요가']` 로 적어 두고(가운데 이름에 **작은따옴표가 들어 있어** 파이썬에서는 큰따옴표로 감싸야 합니다), `for` 로 돌며 `run_cypher(class_merge_q, name=이름)` 으로 실행하세요(**인자 이름도 `name`** 이어야 자리표시자와 이어집니다).
- 이어서 모든 `Class` 의 이름을 별칭 **`name`** 으로 조회해 **정렬한 리스트**를 변수 **`class_names`** 에 담아 출력하세요.

**예시**: 세 이름을 넣었지만 **새로 생기는 것은 스피닝 하나뿐**입니다. 모닝필라테스는 12번에서, 하타요가는 시드가 이미 만들었고 `MERGE` 는 있는 것을 그대로 두기 때문입니다. `class_names` 는 **7개** 입니다.

**주의**: 값을 문자열에 이어 붙이지 마세요(`"MERGE (:Class {name: '" + cn + "'})"`). **가운데 이름에 작은따옴표가 들어 있어서**, 이어 붙이면 그 따옴표가 쿼리의 따옴표와 엉켜 `CypherSyntaxError` 로 실제로 깨집니다. 자리표시자를 쓰면 값은 값으로만 전달돼 그런 일이 없습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 쿼리는 한 번만 쓰고, 바뀌는 값만 run_cypher 의 이름 붙인 인자로 넘긴다.
- 자리표시자 이름($name)과 인자 이름(name=)이 같아야 값이 그 자리에 꽂힌다.

세부구현:
1. "MERGE (:Class {name: $name})" 을 class_merge_q 에 담는다.
2. new_classes 리스트를 만들고 for 로 돌며 run_cypher(class_merge_q, name=cn) 을 실행한다.
3. 모든 Class 를 MATCH 해 이름을 별칭 name 으로 RETURN 하고, 정렬해 class_names 에 담아 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert '$name' in class_merge_q, (
    'class_merge_q 에 자리표시자 $name 이 없습니다. 값을 문자열에 이어 붙이지 말고 쿼리에는 '
    '$name 을 두고 run_cypher 의 name= 인자로 넘기세요'
)
assert 'MERGE' in class_merge_q.upper(), (
    'class_merge_q 는 MERGE 문이어야 합니다. CREATE 로 쓰면 이미 있는 하타요가가 하나 더 생깁니다'
)
assert sorted(new_classes) == ["Rock'n'Roll 스피닝", '모닝필라테스', '하타요가'], (
    'new_classes 에 세 이름을 리스트로 담으세요'
)
for _cn in new_classes:
    run_cypher(class_merge_q, name=_cn)   # 채점이 같은 루프를 한 번 더: MERGE 면 하나도 늘지 않는다
assert sorted(r['name'] for r in run_cypher("MATCH (c:Class) RETURN c.name AS name")) == ["Rock'n'Roll 스피닝", '모닝필라테스', '상체근력', '트레일런', '파워리프팅', '플라잉요가', '하타요가'], (
    '수업 목록이 다릅니다. MERGE 로 적재했는지(CREATE 면 재실행에서 늘어납니다), 12번을 먼저 '
    '풀었는지 확인하세요'
)
assert class_names == ["Rock'n'Roll 스피닝", '모닝필라테스', '상체근력', '트레일런', '파워리프팅', '플라잉요가', '하타요가'], (
    'class_names 에는 모든 Class 이름을 별칭 name 으로 조회해 정렬한 리스트를 담으세요'
)
print('✅ 통과!')

---
수고했어요! LV2 에서 다단 패턴·방향·조건 조합·관계 속성·MERGE 등록에 더해, 조건을 뒤집는 `NOT`, 중복을 없애는 `DISTINCT`, 줄을 세우고 끊는 `ORDER BY`·`LIMIT`, 처음과 재확인을 가르는 `ON CREATE SET`·`ON MATCH SET`, 값을 따로 넘기는 `$이름` 자리표시자까지 **조합**해 다뤘습니다.